# Framing Analysis Institutions

Notebook for institution-focused framing analysis.

In [ ]:
import re
from pathlib import Path

import pandas as pd

# Load original dataframe if not already in memory
if "df" not in globals():
    df = pd.read_csv("/Users/katinkakurz/projects/Thesis/data/raw/df_combined.csv")

# Keep only the columns we need and drop Tagesschau rows
search_df = df[["row_id", "source", "Title", "Text"]].copy()
search_df = search_df[
    search_df["source"].fillna("").astype(str).str.casefold() != "tagesschau"
].copy()

# -----------------------------
# 1. REGEX PATTERNS
# -----------------------------
MEDIA_PATTERNS = [
    r"\bard\b",
    r"\bzdf\b",
    # Exclude "Anti-Spiegel" / "Anti Spiegel"
    r"\b(?:der\s+)?(?<!anti-)(?<!anti\s)spiegel\b",
    r"\btagesschau(?:\.de)?\b",
    r"\btagesschau24\b",
    r"\breuters\b",
    r"\bjan\s+böhmermann\b",
    r"\bndr(?:\s+info)?\b",
    r"\bnorddeutscher\s+rundfunk\b",
    r"\bbild-zeitung\b|\bdie\s+bild\b",
    r"\bpolitico\b",
    r"\bswr\b",
    r"\bsüdwestrundfunk\b",
    r"\bdpa\b",
    r"\bwdr\b",
    r"\bwestdeutscher\s+rundfunk\b",
    r"\bfaz\b",
    r"\bfrankfurter\s+allgemeine(?:n)?\s+zeitung\b",
    r"\börr\b",
    r"\bhandelsblatt\b",
    r"\bdeutschlandfunk\b",
    r"\btagesspiegel\b",
    r"\btaz\b",
    r"\bbr\b",
    r"\bbayerischer\s+rundfunk\b",
    r"\bberliner\s+zeitung\b",
    r"\brbb\b",
    r"\brundfunk\s+berlin-brandenburg\b",
    r"\bcorrectiv\b",
    r"\bsz\b",
    r"\bsüddeutsch(?:e|en)\s+zeitung\b",
    r"\bstern\b",
    r"\bdas\s+erste\b",
    r"\bthe\s+european\b",
    r"\brtl\b",
    r"\bmdr\b",
    r"\bmitteldeutscher\s+rundfunk\b",
    r"\brnd\b",
    r"\bredaktionsnetzwerk\s+deutschland\b",
    r"\bcaren\s+miosga\b",
    r"\brheinisch(?:e|en)\s+post\b",
    # r"\bbund\b",
    r"\b(?:markus\s+)?lanz\b",
    # r"\bfocus\b",
    r"\beuronews\b",
]

# KEY CHANGE:
# "Medien" alone should NOT match.
# Only compounds like "Staatsmedien", "Leitmedien", "Altmedien" should match.
#KEYWORD_PATTERNS #= [
#     r"[a-zäöüß-]*presse[a-zäöüß-]*",
#     r"[a-zäöüß-]*funk[a-zäöüß-]*",
#     r"[a-zäöüß-]+(?:medien|medium|medial)[a-zäöüß-]*",
#     r"[a-zäöüß-]*sender[a-zäöüß-]*",
#     r"[a-zäöüß-]*öffentlich[a-zäöüß-]*[ -]?rechtlich[a-zäöüß-]*",
#     r"[a-zäöüß-]*fernseh[a-zäöüß-]*",
# ]

MASTER_PATTERN = re.compile(
    "|".join(f"(?:{p})" for p in MEDIA_PATTERNS),
    flags=re.IGNORECASE,
)

# -----------------------------
# 2. HELPER FUNCTIONS
# -----------------------------
def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def combine_title_text(title, text):
    title = normalize_text(title)
    text = normalize_text(text)

    if title and text and not re.search(r"[.!?…:;]$", title):
        title = title + "."

    return f"{title} {text}".strip()

def split_sentences(text):
    text = normalize_text(text)
    if not text:
        return []

    sentences = re.split(r"(?<=[.!?…])\s+(?=[A-ZÄÖÜ0-9\"'“„(])", text)
    sentences = [s.strip() for s in sentences if s and s.strip()]
    return sentences

def extract_context_windows(text, pattern, window=1):
    sentences = split_sentences(text)
    if not sentences:
        return []

    matched_indices = [
        i for i, sentence in enumerate(sentences)
        if pattern.search(sentence)
    ]

    if not matched_indices:
        return []

    # Merge overlapping / immediately adjacent windows
    merged_windows = []
    for idx in matched_indices:
        start = max(0, idx - window)
        end = min(len(sentences), idx + window + 1)

        if merged_windows and start <= merged_windows[-1][1]:
            merged_windows[-1] = (merged_windows[-1][0], max(merged_windows[-1][1], end))
        else:
            merged_windows.append((start, end))

    return [" ".join(sentences[start:end]) for start, end in merged_windows]

# -----------------------------
# 3. BUILD SEARCH TEXT
# -----------------------------
search_df["combined_text"] = [
    combine_title_text(title, text)
    for title, text in zip(search_df["Title"], search_df["Text"])
]

candidate_df = search_df[
    search_df["combined_text"].str.contains(MASTER_PATTERN, na=False)
].copy()

# Add explicit ÖRR/media-related keywords (exact terms)
EXTRA_KEYWORDS = [
    "Mainstreammedien",
    "Staatsmedien",
    "Staatsfunk",
    "Qualitätsmedien",
    "Staatssender",
    "Lügenpresse",
    "Haltungsjournalisten",
    "Gleichschaltung",
    "Mainstreampresse",
    "Altmedien",
    "Systemmedien",
    "Qualitätsjournalismus",
    "Alternativmedien",
    "Gesternmedien",
    "Haltungsmedien",
    "Westmedien",
    "Regierungsmedien",
    "Linkspresse",
    "Qualitätspresse",
    "Haltungsjournalismus",
    "Propagandamedien",
    "Propagandasender",
    "Propagandamaschine",
    "Medienpropaganda",
    "Staatsrundfunk",
]

EXTRA_KEYWORD_PATTERNS = [rf"\b{re.escape(k)}\b" for k in EXTRA_KEYWORDS]

MASTER_PATTERN = re.compile(
    "|".join(f"(?:{p})" for p in MEDIA_PATTERNS + EXTRA_KEYWORD_PATTERNS),
    flags=re.IGNORECASE,
)

candidate_df = search_df[
    search_df["combined_text"].str.contains(MASTER_PATTERN, na=False)
].copy()

print(f"Candidate articles with at least one hit (excluding Tagesschau): {len(candidate_df):,}")

# -----------------------------
# 4. EXTRACT 1-SENTENCE CONTEXT WINDOWS
# -----------------------------
rows = []

for row in candidate_df.itertuples(index=False):
    windows = extract_context_windows(row.combined_text, MASTER_PATTERN, window=1)

    for context in windows:
        rows.append(
            {
                "row_id": row.row_id,
                "source": row.source,
                "Title": row.Title,
                "Text": row.Text,
                "context_window": context,
            }
        )

media_context_df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

print(f"Context windows extracted: {len(media_context_df):,}")
display(media_context_df.head())

# -----------------------------
# 5. ONE ROW PER ARTICLE
# -----------------------------
media_article_df = (
    media_context_df.groupby(["row_id", "source", "Title","Text"], as_index=False)
    .agg(context_window=("context_window", "\n\n---\n\n".join))
)

print(f"Unique articles in filtered set: {len(media_article_df):,}")
display(media_article_df.head())

KeyError: "['Text'] not in index"